<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.3-embeddings-and-vector-stores/notebooks/exercises-6_3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.3: Embeddings & Vector Stores

*Module 6 · Retrieval-Augmented Generation (RAG)*

> In Lesson 6.1 we stored vectors in a Python list. That works for 100 documents. What about 10 million? This lesson covers production embedding APIs (Gemini, open-source), and three vector databases: ChromaDB (local prototyping), AlloyDB pgvector (GCP managed), and Vertex AI Vector Search (billion-scale). Each coded with a working demo.

`Vertex AI Embeddings` · `ChromaDB` · `pgvector` · `Vector Search` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-6.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Embedding APIs — Text to Vectors — `01_embeddings.py`
2. Step 2 — ChromaDB — Local Vector Store for Prototyping — `02_chromadb.py`
3. Step 3 — AlloyDB pgvector — SQL + Vectors in One Database — `03_pgvector.py`
4. Step 4 — Vertex AI Vector Search — Billion-Scale Production — `04_vertex_vector_search.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy chromadb sentence-transformers


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Embedding APIs — Text to Vectors

Gemini text-embedding-004 vs open-source sentence-transformers. Compare quality and speed.

**`01_embeddings.py`** · _Block 1 of 4_

EMBEDDING APIs — Gemini vs Open-Source


In [ ]:
# EMBEDDING APIs — Gemini vs Open-Source
import google.generativeai as genai
import numpy as np, os, time

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Gemini Embedding (API)
def gemini_embed(texts):
    results = []
    for t in texts:
        r = genai.embed_content(model="models/text-embedding-004", content=t)
        results.append(np.array(r["embedding"]))
    return results

# Open-source embedding (local)
# from sentence_transformers import SentenceTransformer
# local_model = SentenceTransformer("all-MiniLM-L6-v2")
# def local_embed(texts): return local_model.encode(texts)

texts = ["machine learning algorithms", "AI and deep learning models",
         "cooking biryani recipe", "refund and return policy"]

print("Embedding Comparison:\n")
start = time.time()
embs = gemini_embed(texts)
elapsed = (time.time()-start)*1000

print(f"  Gemini text-embedding-004:")
print(f"    Dimensions: {len(embs[0])}")
print(f"    Latency: {elapsed:.0f}ms for {len(texts)} texts")
print(f"    Cost: $0.004 per 1K embeddings\n")

# Similarity matrix
print("  Similarity matrix:")
for i, t1 in enumerate(texts):
    row = f"    {t1[:20]:20s}"
    for j, t2 in enumerate(texts):
        sim = np.dot(embs[i],embs[j])/(np.linalg.norm(embs[i])*np.linalg.norm(embs[j]))
        row += f" {sim:.2f}"
    print(row)

print(f"\n  ML ↔ AI: high similarity. ML ↔ biryani: low. That is semantic search.")

print(f"\n  Embedding model comparison:")
print(f"    {'Model':30s} {'Dims':>5s} {'Cost':>12s} {'Where'}")
print(f"    {'Gemini text-embedding-004':30s} {'768':>5s} {'$0.004/1K':>12s} API")
print(f"    {'OpenAI text-embedding-3-small':30s} {'1536':>5s} {'$0.02/1K':>12s} API")
print(f"    {'all-MiniLM-L6-v2':30s} {'384':>5s} {'Free':>12s} Local")
print(f"    {'BGE-large-en':30s} {'1024':>5s} {'Free':>12s} Local")


> **What just happened?** Gemini produced 768-dim embeddings. "ML algorithms" and "AI deep learning" scored high similarity (~0.8). "ML" and "biryani" scored low (~0.2). Key decision: Gemini embeddings for production (fast, cheap API). Local models (MiniLM, BGE) for privacy-sensitive data or offline use. Gemini at $0.004/1K is 5x cheaper than OpenAI.


### Step 2 · ChromaDB — Local Vector Store for Prototyping

pip install, 5 lines to index, 3 lines to search. Perfect for development.

**`02_chromadb.py`** · _Block 2 of 4_

CHROMADB — Local vector database — pip install chromadb


In [ ]:
# CHROMADB — Local vector database
# pip install chromadb
import chromadb

client = chromadb.Client()  # in-memory (or PersistentClient for disk)
collection = client.create_collection("netsetos_docs")

# Index documents (Chroma auto-embeds with default model)
docs = [
    "Netsetos offers GenAI courses in Hyderabad. 14 modules, 146 hours.",
    "Refund policy: full refund within 7 days. 50% from 8-30 days.",
    "GenAI course costs 14999 rupees. Includes Discord, live sessions, certificate.",
    "Live classes daily at 7 PM IST on YouTube. 85% completion rate.",
    "Prerequisites: basic Python and high school math. No ML experience needed.",
]

collection.add(
    documents=docs,
    ids=[f"doc_{i}" for i in range(len(docs))],
    metadatas=[{"source": f"faq_{i}.txt"} for i in range(len(docs))],
)

print(f"Indexed: {collection.count()} documents\n")

# Search by natural language
for q in ["How much does the course cost?", "Can I get a refund?", "Do I need ML experience?"]:
    results = collection.query(query_texts=[q], n_results=2)
    print(f"  Q: {q}")
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"    [{dist:.3f}] {doc[:70]}...")
    print()

print("ChromaDB: 5 lines to index, 3 to search. Perfect for prototyping.")
print("For production: switch to Vertex AI Vector Search or AlloyDB.")


### Step 3 · AlloyDB pgvector — SQL + Vectors in One Database

If you already use PostgreSQL, add vector search without a separate database.

**`03_pgvector.py`** · _Block 3 of 4_

ALLOYDB PGVECTOR — SQL + vector search — pip install asyncpg pgvector sqlalchemy


In [ ]:
# ALLOYDB PGVECTOR — SQL + vector search
# pip install asyncpg pgvector sqlalchemy
# Requires AlloyDB instance on GCP

# Schema: documents table with vector column
CREATE_SQL = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS documents (
    id SERIAL PRIMARY KEY,
    content TEXT NOT NULL,
    source VARCHAR(255),
    embedding vector(768),  -- Gemini embedding dimensions
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE INDEX ON documents 
    USING ivfflat (embedding vector_cosine_ops) 
    WITH (lists = 100);
"""

# Insert with embedding
INSERT_SQL = """
INSERT INTO documents (content, source, embedding) 
VALUES ($1, $2, $3::vector)
"""

# Search by similarity
SEARCH_SQL = """
SELECT content, source, 
       1 - (embedding <=> $1::vector) AS similarity
FROM documents
ORDER BY embedding <=> $1::vector
LIMIT $2
"""

print("AlloyDB pgvector Setup:\n")
print("  1. CREATE EXTENSION vector;")
print("  2. Add vector(768) column to your existing table")
print("  3. CREATE INDEX using ivfflat for fast search")
print("  4. Search: ORDER BY embedding <=> query_vector\n")
print("  Advantages:")
print("    - Same database for structured data AND vectors")
print("    - SQL joins: combine vector search with filters")
print("    - AlloyDB: GCP-managed, auto-scaling, 100x faster than vanilla PG")
print("    - Example: WHERE category='genai' ORDER BY embedding <=> query")


### Step 4 · Vertex AI Vector Search — Billion-Scale Production

Google’s managed vector search. Sub-10ms at billion scale. ScaNN algorithm.

**`04_vertex_vector_search.py`** · _Block 4 of 4_

VERTEX AI VECTOR SEARCH — Billion-scale — pip install google-cloud-aiplatform


In [ ]:
# VERTEX AI VECTOR SEARCH — Billion-scale
# pip install google-cloud-aiplatform
from google.cloud import aiplatform

# Initialize
aiplatform.init(project="your-project", location="us-central1")

# Step 1: Create Index
index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name="netsetos-rag-index",
    dimensions=768,             # Gemini embedding dims
    approximate_neighbors_count=10,
    distance_measure_type="COSINE_DISTANCE",
)

# Step 2: Create Endpoint
endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name="netsetos-rag-endpoint",
    public_endpoint_enabled=True,
)

# Step 3: Deploy Index to Endpoint
endpoint.deploy_index(index=index, deployed_index_id="rag_deployed")

# Step 4: Query
# response = endpoint.find_neighbors(
#     deployed_index_id="rag_deployed",
#     queries=[query_embedding],
#     num_neighbors=5,
# )

print("Vertex AI Vector Search:\n")
print("  Scale: billions of vectors")
print("  Latency: <10ms at 1B vectors")
print("  Algorithm: ScaNN (Google Research)")
print("  Cost: ~$0.50/hr per deployed endpoint\n")
print("  When to use:")
print("    ChromaDB:     <1M docs, prototyping, local dev")
print("    pgvector:     1M-10M, need SQL joins, existing Postgres")
print("    Vector Search: 10M+, sub-10ms required, production scale")


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
